# Build Index — Combined Phase 1 + Phase 2
**Multimodal RAG for E-commerce**

Run this top to bottom: **Runtime → Run all**. It does the whole offline pipeline in ONE runtime —
Phase 1 cleans the catalog and caches images, Phase 2 encodes with CLIP and builds the ChromaDB index.

**Before running:** (1) Runtime → Change runtime type → **T4 GPU**.  (2) Click the folder icon in the left
sidebar and upload **`amazon_products.csv`**.


In [ ]:
# Install dependencies. transformers is pinned below 5.0 ON PURPOSE:
# transformers 5.x changed CLIP get_text_features/get_image_features to return an
# object instead of a tensor, which breaks this pipeline. 4.x returns a tensor.
!pip install -q "transformers<5" torch torchvision pillow pandas pyarrow chromadb tqdm requests

# Phase 1 — Data Preparation
**Multimodal RAG for E-commerce**

Goal of this notebook: turn the raw Amazon Product Dataset 2020 into a clean, deduplicated subset with:
- One row per product
- 2–3 candidate product description strings (for ablation in Phase 3)
- Cached local images (primary image per product, resized for CLIP)
- A stable `product_id` we can use as the key in ChromaDB

Run cells in order, but feel free to pause after the audit cells to adjust filters before continuing.

> Note on column names: this notebook assumes the schema of the promptcloud version on Kaggle (Title Case with spaces — `Product Name`, `Brand Name`, etc.). If your CSV uses different names, the audit cell will surface them and you can rename.

## 1. Setup

In [ ]:
# In Colab, uncomment to install:
# !pip install -q pandas pyarrow pillow requests tqdm

import os
import re
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
pd.set_option('display.max_colwidth', 100)

# Paths — adjust to wherever your CSV lives
DATA_PATH = "amazon_products.csv"
OUTPUT_DIR = Path("./prepared_data")
IMAGES_DIR = OUTPUT_DIR / "images"
OUTPUT_DIR.mkdir(exist_ok=True)
IMAGES_DIR.mkdir(exist_ok=True)

# Tuning knobs
TARGET_SIZE = 2000       # final product count after sampling
MAX_IMAGE_SIDE = 512      # cached image resize (CLIP needs 224, 512 keeps headroom)
URL_CHECK_SAMPLE = 200    # how many URLs to test for the broken-rate estimate

## 2. Load and inspect schema

First look at what we actually have. Don't trust any documentation — eyeball the columns yourself.

In [ ]:
df = pd.read_csv(DATA_PATH)
print(f"Shape: {df.shape}")
print(f"Memory: {df.memory_usage(deep=True).sum() / 1e6:.1f} MB")
print(f"\nColumns ({len(df.columns)}):")
for c in df.columns:
    print(f"  {c}")
df.head(2)

## 3. Coverage audit

For each column: how many non-null values, how unique, and a sample. This is the most important cell — it tells you which fields are actually usable.

In [ ]:
audit = pd.DataFrame({
    'dtype': df.dtypes.astype(str),
    'non_null': df.notna().sum(),
    'null_pct': (df.isna().sum() / len(df) * 100).round(1),
    'unique': df.nunique(),
    'sample': [str(df[c].dropna().iloc[0])[:80] if df[c].notna().any() else None for c in df.columns],
})
audit.sort_values('null_pct')

**Decision point:** look at the audit above. Which fields have high coverage AND are useful for a product description? Common winners: `Product Name`, `Brand Name`, `Category`, `Selling Price`, `About Product`. The `Image` column is critical — if its coverage is low, you have a much smaller usable dataset than the shape suggests.

## 4. Drop rows missing critical fields

In [ ]:
required = ['Product Name', 'Image']           # must have a title AND at least one image URL
recommended = ['Brand Name', 'Category', 'About Product', 'Selling Price']

df_clean = df.dropna(subset=required).copy()
print(f"After dropping rows missing {required}: {len(df_clean):,} of {len(df):,} ({len(df_clean)/len(df)*100:.1f}%)")
print(f"\nCoverage of recommended fields in cleaned set:")
for col in recommended:
    if col in df_clean.columns:
        pct = df_clean[col].notna().sum() / len(df_clean) * 100
        print(f"  {col}: {pct:.1f}%")

## 5. Deduplicate

Amazon catalogs often repeat the same product across rows (variants, multiple listings). Dedup by `(Product Name, Brand Name)` — strict enough to catch real dupes, loose enough to keep legitimate variants when they have different names.

In [ ]:
before = len(df_clean)
df_clean = df_clean.drop_duplicates(subset=['Product Name', 'Brand Name'], keep='first')
print(f"Dropped {before - len(df_clean):,} duplicate rows")
print(f"Unique products: {len(df_clean):,}")

## 6. Category distribution

The `Category` field is usually a `|`-separated path like `Toys & Games | Puzzles | Jigsaw Puzzles`. Take the top-level for stratified sampling later.

In [ ]:
df_clean['top_category'] = df_clean['Category'].astype(str).str.split('|').str[0].str.strip()
top_cats = df_clean['top_category'].value_counts().head(20)
print(top_cats)

ax = top_cats.plot(kind='barh', figsize=(8, 6), title='Top 20 categories in cleaned set')
ax.invert_yaxis()

## 7. Build candidate description strings

Build three variants of the description so you can ablate in Phase 3 and see which gives the best retrieval. Important caveat: **CLIP's text encoder maxes out at 77 tokens (~50–60 words)**. The `full` variant will mostly be truncated — keep it for comparison but expect `standard` to be the winner in practice.

In [ ]:
def safe(v):
    return str(v) if pd.notna(v) and str(v).strip() else None

def build_desc_minimal(row):
    parts = [safe(row.get('Product Name')), safe(row.get('Brand Name'))]
    return ' | '.join(p for p in parts if p)

def build_desc_standard(row):
    parts = [
        safe(row.get('Product Name')),
        f"Brand: {row['Brand Name']}" if safe(row.get('Brand Name')) else None,
        f"Category: {row['top_category']}" if safe(row.get('top_category')) else None,
        f"Price: {row['Selling Price']}" if safe(row.get('Selling Price')) else None,
        safe(row.get('About Product')),
    ]
    return ' | '.join(p for p in parts if p)

def build_desc_full(row):
    base = build_desc_standard(row)
    extras = []
    for col, label in [('Technical Details', 'Specs'), ('Product Specification', 'Spec')]:
        if safe(row.get(col)):
            extras.append(f"{label}: {row[col]}")
    return base + (' | ' + ' | '.join(extras) if extras else '')

df_clean['desc_minimal'] = df_clean.apply(build_desc_minimal, axis=1)
df_clean['desc_standard'] = df_clean.apply(build_desc_standard, axis=1)
df_clean['desc_full'] = df_clean.apply(build_desc_full, axis=1)

print("Description length stats (characters):")
for col in ['desc_minimal', 'desc_standard', 'desc_full']:
    chars = df_clean[col].str.len()
    print(f"  {col}: mean={chars.mean():.0f}, median={chars.median():.0f}, p95={chars.quantile(0.95):.0f}, max={chars.max():.0f}")

print("\nExample (random product):")
sample = df_clean.sample(1, random_state=7).iloc[0]
for col in ['desc_minimal', 'desc_standard', 'desc_full']:
    print(f"\n[{col}]\n{sample[col][:300]}")

## 8. Parse and inspect image URLs

In [ ]:
def parse_image_urls(img_str):
    if pd.isna(img_str):
        return []
    return [u.strip() for u in str(img_str).split('|') if u.strip().startswith('http')]

df_clean['image_urls'] = df_clean['Image'].apply(parse_image_urls)
df_clean['num_images'] = df_clean['image_urls'].apply(len)

print(f"Images per product: mean={df_clean['num_images'].mean():.1f}, max={df_clean['num_images'].max()}")
print(f"Products with 0 valid URLs: {(df_clean['num_images'] == 0).sum()}")

df_clean = df_clean[df_clean['num_images'] > 0].copy()
df_clean['primary_image_url'] = df_clean['image_urls'].apply(lambda urls: urls[0])
print(f"\nAfter image-URL filter: {len(df_clean):,} products")

## 9. Estimate broken-URL rate

This dataset is from 2020. By 2026, some Amazon CDN URLs may have expired. Test a sample to know what you're working with before downloading thousands.

In [ ]:
import requests
from concurrent.futures import ThreadPoolExecutor

def check_url(url, timeout=5):
    try:
        r = requests.head(url, timeout=timeout, allow_redirects=True)
        return r.status_code < 400
    except Exception:
        return False

sample_urls = df_clean['primary_image_url'].sample(min(URL_CHECK_SAMPLE, len(df_clean)), random_state=42).tolist()
with ThreadPoolExecutor(max_workers=20) as ex:
    results = list(ex.map(check_url, sample_urls))

broken_rate = 1 - sum(results) / len(results)
print(f"Estimated broken-URL rate: {broken_rate*100:.1f}% (tested {len(results)} URLs)")
print(f"At your target of {TARGET_SIZE}, expect ~{int(TARGET_SIZE * broken_rate)} dead links.")

## 10. Stratified sample to target size

If you have more products than TARGET_SIZE, sample with category stratification so you don't end up with a dataset that's 80% "Toys & Games".

In [ ]:
if len(df_clean) > TARGET_SIZE:
    # Proportional allocation per category, floor of 30 per category to keep small ones representable
    counts = df_clean['top_category'].value_counts()
    alloc = (counts * TARGET_SIZE / counts.sum()).round().astype(int).clip(lower=30)
    
    df_sample = (
        df_clean.groupby('top_category', group_keys=False)
        .apply(lambda g: g.sample(min(len(g), alloc.get(g.name, 30)), random_state=42))
    )
    df_sample = df_sample.sample(min(TARGET_SIZE, len(df_sample)), random_state=42).reset_index(drop=True)
else:
    df_sample = df_clean.copy().reset_index(drop=True)

df_sample['product_id'] = ['prod_' + str(i).zfill(6) for i in range(len(df_sample))]
print(f"Sampled {len(df_sample):,} products across {df_sample['top_category'].nunique()} categories")
print("\nTop 10 categories in sample:")
print(df_sample['top_category'].value_counts().head(10))

## 11. Cache primary images locally

Download the primary image for each sampled product. Resize to 512px max side, save as JPEG. This takes the longest — depending on the broken-URL rate, plan 5–15 minutes for 10K products. Cells are idempotent (already-downloaded images are skipped) so you can re-run safely.

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from PIL import Image
from io import BytesIO

def download_image(args):
    product_id, url = args
    target = IMAGES_DIR / f"{product_id}.jpg"
    if target.exists() and target.stat().st_size > 0:
        return product_id, True, "cached"
    try:
        r = requests.get(url, timeout=10)
        if r.status_code != 200:
            return product_id, False, f"http_{r.status_code}"
        img = Image.open(BytesIO(r.content)).convert("RGB")
        img.thumbnail((MAX_IMAGE_SIDE, MAX_IMAGE_SIDE), Image.LANCZOS)
        img.save(target, "JPEG", quality=85)
        return product_id, True, "ok"
    except Exception as e:
        return product_id, False, type(e).__name__

tasks = list(zip(df_sample['product_id'], df_sample['primary_image_url']))
results = []

with ThreadPoolExecutor(max_workers=16) as ex:
    futures = [ex.submit(download_image, t) for t in tasks]
    for i, f in enumerate(as_completed(futures)):
        results.append(f.result())
        if (i + 1) % 500 == 0:
            ok = sum(1 for _, success, _ in results if success)
            print(f"  {i+1}/{len(tasks)} — {ok} success, {i+1-ok} failed")

ok_count = sum(1 for _, success, _ in results if success)
print(f"\nDone: {ok_count}/{len(tasks)} images cached ({ok_count/len(tasks)*100:.1f}%)")

# Failure breakdown
from collections import Counter
failures = Counter(reason for _, success, reason in results if not success)
if failures:
    print(f"\nFailure reasons: {dict(failures.most_common(5))}")

## 12. Filter to successful downloads, then save

Drop any rows whose image didn't make it to disk — we need both modalities for every product in the index.

In [ ]:
success_ids = {pid for pid, success, _ in results if success}
df_final = df_sample[df_sample['product_id'].isin(success_ids)].copy()
df_final['local_image_path'] = df_final['product_id'].apply(lambda p: str(IMAGES_DIR / f"{p}.jpg"))
print(f"Final dataset: {len(df_final):,} products with both description AND cached image")

keep_cols = [
    'product_id',
    'Product Name', 'Brand Name', 'top_category',
    'Selling Price', 'About Product',
    'desc_minimal', 'desc_standard', 'desc_full',
    'primary_image_url', 'local_image_path',
]
keep_cols = [c for c in keep_cols if c in df_final.columns]

df_final[keep_cols].to_parquet(OUTPUT_DIR / "products_cleaned.parquet", index=False)
df_final[keep_cols].to_csv(OUTPUT_DIR / "products_cleaned.csv", index=False)

print(f"\nSaved:")
print(f"  {OUTPUT_DIR / 'products_cleaned.parquet'}")
print(f"  {OUTPUT_DIR / 'products_cleaned.csv'}")
print(f"  {IMAGES_DIR}/ ({len(list(IMAGES_DIR.glob('*.jpg')))} images)")

## 13. Spot check

Look at 5 random products with their description and image side by side. Catch any garbage descriptions or wrong images before they go into the vector store.

In [ ]:
import matplotlib.pyplot as plt

samples = df_final.sample(5, random_state=42)
fig, axes = plt.subplots(1, 5, figsize=(20, 5))
for ax, (_, row) in zip(axes, samples.iterrows()):
    img = Image.open(row['local_image_path'])
    ax.imshow(img)
    ax.set_title(row['Product Name'][:50] + '...', fontsize=9)
    ax.axis('off')
plt.tight_layout()
plt.show()

for _, row in samples.iterrows():
    print(f"\n{row['product_id']} | {row['top_category']}")
    print(f"  desc_standard: {row['desc_standard'][:250]}")

## What's next — Phase 2

You now have:
- `prepared_data/products_cleaned.parquet` — one row per product with three description variants
- `prepared_data/images/` — primary product images, resized for CLIP

Phase 2 picks up here: load CLIP (using the same `openai/clip-vit-base-patch32` model from the hands-on notebook), encode each product's `desc_standard` and its image, normalize the embeddings, and write them to ChromaDB keyed by `product_id`. The Dr. Bousetouane notebook's `get_text_features` / `get_image_features` calls are exactly the primitives you'll batch up.

---
# Phase 2 — CLIP Encoding & Vector Indexing


# Phase 2 — CLIP Encoding & Vector Indexing
**Multimodal RAG for E-commerce**

Goal: take the cleaned product catalog from Phase 1 and turn it into a vector database.

What this notebook does:
1. Load the cleaned products (parquet from Phase 1)
2. Load CLIP (`openai/clip-vit-base-patch32`) on GPU if available
3. Batch-encode every product's text description (using `desc_standard` as the default)
4. Batch-encode every product's primary image
5. L2-normalize all embeddings (so cosine similarity = dot product)
6. Write both text and image embeddings to ChromaDB with product metadata
7. Sanity-check the index with a couple of test queries

Why two collections (one for text, one for image)? Keeping them separate lets you query either modality independently and ablate later. Both embeddings live in the same 512-dim CLIP space, so a text query can still match against image embeddings if you want — but most retrieval recipes query text-against-text and image-against-image, then combine.

## 1. Setup

In [ ]:
# In Colab:
# (dependencies already installed in the first cell above)

import os
import torch
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image
from tqdm.auto import tqdm

from transformers import CLIPModel, CLIPProcessor
import chromadb

# Paths
DATA_DIR = Path("./prepared_data")
PRODUCTS_PARQUET = DATA_DIR / "products_cleaned.parquet"
IMAGES_DIR = DATA_DIR / "images"
CHROMA_DIR = Path("./chroma_db")
CHROMA_DIR.mkdir(exist_ok=True)

# Tuning
MODEL_NAME = "openai/clip-vit-base-patch32"   # ViT-L/14 if you want better quality at ~3x cost
BATCH_SIZE = 64                                # Lower if you run out of GPU memory
DESC_FIELD = "desc_standard"                  # Which description variant to index (minimal/standard/full)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. Load the cleaned products

In [ ]:
df = pd.read_parquet(PRODUCTS_PARQUET)
print(f"Loaded {len(df):,} products")
print(f"Columns: {list(df.columns)}")
df.head(2)

## 3. Load CLIP

In [ ]:
print(f"Loading {MODEL_NAME}...")
model = CLIPModel.from_pretrained(MODEL_NAME).to(DEVICE)
model.eval()
processor = CLIPProcessor.from_pretrained(MODEL_NAME)
print(f"Loaded. Text encoder max length: {processor.tokenizer.model_max_length} tokens")

## 4. Batch-encode text descriptions

CLIP's text encoder truncates at 77 tokens. Anything longer just gets cut off — the truncation is silent, so don't be surprised if very long descriptions don't outperform shorter ones.

In [ ]:
@torch.no_grad()
def encode_texts(texts, batch_size=BATCH_SIZE):
    embeddings = []
    for i in tqdm(range(0, len(texts), batch_size), desc="Encoding text"):
        batch = texts[i:i+batch_size]
        inputs = processor(text=batch, return_tensors="pt", padding=True, truncation=True, max_length=77)
        inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
        feats = model.get_text_features(**inputs)
        feats = feats.pooler_output if hasattr(feats, "pooler_output") else feats
        embeddings.append(feats.cpu().numpy())
    return np.vstack(embeddings)

texts = df[DESC_FIELD].fillna("").tolist()
text_embeddings = encode_texts(texts)
print(f"Text embeddings shape: {text_embeddings.shape}")

## 5. Batch-encode product images

In [ ]:
@torch.no_grad()
def encode_images(image_paths, batch_size=BATCH_SIZE):
    embeddings = []
    failed_indices = []
    for i in tqdm(range(0, len(image_paths), batch_size), desc="Encoding images"):
        batch_paths = image_paths[i:i+batch_size]
        batch_images = []
        valid_indices = []
        for j, p in enumerate(batch_paths):
            try:
                img = Image.open(p).convert("RGB")
                batch_images.append(img)
                valid_indices.append(i + j)
            except Exception as e:
                failed_indices.append(i + j)
        if not batch_images:
            embeddings.append(np.zeros((len(batch_paths), 512), dtype=np.float32))
            continue
        inputs = processor(images=batch_images, return_tensors="pt")
        inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
        feats = model.get_image_features(**inputs)
        feats = (feats.pooler_output if hasattr(feats, "pooler_output") else feats).cpu().numpy()
        # Reconstruct full batch shape (zeros for failed images — we'll drop them later)
        batch_embs = np.zeros((len(batch_paths), feats.shape[1]), dtype=np.float32)
        for k, vi in enumerate([idx - i for idx in valid_indices]):
            batch_embs[vi] = feats[k]
        embeddings.append(batch_embs)
    return np.vstack(embeddings), failed_indices

image_paths = df["local_image_path"].tolist()
image_embeddings, failed = encode_images(image_paths)
print(f"Image embeddings shape: {image_embeddings.shape}")
print(f"Failed to encode: {len(failed)} images")

## 6. L2-normalize embeddings

In [ ]:
def l2_normalize(x):
    norms = np.linalg.norm(x, axis=1, keepdims=True)
    norms = np.where(norms == 0, 1e-12, norms)
    return x / norms

text_embeddings = l2_normalize(text_embeddings).astype(np.float32)
image_embeddings = l2_normalize(image_embeddings).astype(np.float32)
print("Normalized. Sanity check — first vector norm should be 1.0:")
print(f"  Text:  {np.linalg.norm(text_embeddings[0]):.4f}")
print(f"  Image: {np.linalg.norm(image_embeddings[0]):.4f}")

## 7. Drop products whose image encoding failed

In [ ]:
if failed:
    keep_mask = np.ones(len(df), dtype=bool)
    keep_mask[failed] = False
    df = df[keep_mask].reset_index(drop=True)
    text_embeddings = text_embeddings[keep_mask]
    image_embeddings = image_embeddings[keep_mask]
    print(f"After dropping failures: {len(df):,} products")
else:
    print(f"No failures — keeping all {len(df):,} products")

## 8. Initialize ChromaDB and create collections

We create two collections:
- `products_text` — text embeddings, queryable by text
- `products_image` — image embeddings, queryable by image (or by text since both share the CLIP space)

In [ ]:
client = chromadb.PersistentClient(path=str(CHROMA_DIR))

# Clean slate — delete any prior collections with these names
for name in ["products_text", "products_image"]:
    try:
        client.delete_collection(name)
    except Exception:
        pass

text_collection = client.create_collection(
    name="products_text",
    metadata={"hnsw:space": "cosine"}
)
image_collection = client.create_collection(
    name="products_image",
    metadata={"hnsw:space": "cosine"}
)
print(f"Created collections: products_text, products_image")

## 9. Insert embeddings into ChromaDB

In [ ]:
# Build metadata — what we'll store with each vector and return on retrieval
def row_to_metadata(row):
    return {
        "product_id": str(row["product_id"]),
        "product_name": str(row.get("Product Name", ""))[:500],
        "brand": str(row.get("Brand Name", ""))[:200],
        "category": str(row.get("top_category", ""))[:200],
        "price": str(row.get("Selling Price", ""))[:50],
        "about": str(row.get("About Product", ""))[:1000],
        "image_path": str(row.get("local_image_path", "")),
        "image_url": str(row.get("primary_image_url", "")),
        "desc": str(row[DESC_FIELD])[:2000],
    }

ids = df["product_id"].tolist()
metadatas = [row_to_metadata(row) for _, row in df.iterrows()]

# Insert in chunks (ChromaDB has a max batch size)
CHUNK = 1000
for i in tqdm(range(0, len(ids), CHUNK), desc="Inserting text"):
    text_collection.add(
        ids=ids[i:i+CHUNK],
        embeddings=text_embeddings[i:i+CHUNK].tolist(),
        metadatas=metadatas[i:i+CHUNK],
    )

for i in tqdm(range(0, len(ids), CHUNK), desc="Inserting image"):
    image_collection.add(
        ids=ids[i:i+CHUNK],
        embeddings=image_embeddings[i:i+CHUNK].tolist(),
        metadatas=metadatas[i:i+CHUNK],
    )

print(f"\nIndexed {text_collection.count()} text and {image_collection.count()} image embeddings")

## 10. Save embeddings to disk (backup, also useful for eval)

In [ ]:
np.save(DATA_DIR / "text_embeddings.npy", text_embeddings)
np.save(DATA_DIR / "image_embeddings.npy", image_embeddings)
df.to_parquet(DATA_DIR / "products_indexed.parquet", index=False)
print(f"Saved:")
print(f"  text_embeddings.npy   shape={text_embeddings.shape}")
print(f"  image_embeddings.npy  shape={image_embeddings.shape}")
print(f"  products_indexed.parquet  rows={len(df)}")

## 11. Sanity check — quick test queries

In [ ]:
@torch.no_grad()
def embed_text_query(text):
    inputs = processor(text=[text], return_tensors="pt", padding=True, truncation=True, max_length=77)
    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
    feats = model.get_text_features(**inputs)
    feats = (feats.pooler_output if hasattr(feats, "pooler_output") else feats).cpu().numpy()
    return l2_normalize(feats)[0]

test_queries = [
    "wireless bluetooth headphones",
    "kitchen stand mixer",
    "puzzle for kids",
    "running shoes",
]

for q in test_queries:
    q_emb = embed_text_query(q)
    results = text_collection.query(query_embeddings=[q_emb.tolist()], n_results=3)
    print(f"\nQuery: '{q}'")
    for i, meta in enumerate(results["metadatas"][0]):
        print(f"  {i+1}. {meta['product_name'][:80]} ({meta['category']})")

In [ ]:
import chromadb
c = chromadb.PersistentClient(path="chroma_db")
print([(col.name, col.count()) for col in c.list_collections()])

## What's next — Phase 3

The vector DB is ready. Move to `03_retrieval_eval.ipynb` to compute `Recall@1/5/10` across multiple query types — that's the quantitative evaluation the project spec requires.

In [ ]:
# Report versions and package the index for download
import chromadb, transformers, shutil
print('transformers version:', transformers.__version__)
print('chromadb version:    ', chromadb.__version__)
print()
print('>>> In your project requirements.txt, these two lines should read:')
print('    transformers>=4.36,<5')
print(f'    chromadb=={chromadb.__version__}')
print()
shutil.make_archive('chroma_db', 'zip', 'chroma_db')
print('Created chroma_db.zip - right-click it in the file sidebar and choose Download.')